In [6]:
from stable_baselines3 import PPO, TD3
from sb3_contrib import TQC, RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from rl4greencrab import evaluate_agent, multiConstAction, simulator
import pandas as pd
import numpy as np
from rl4greencrab import plot_agent
import ray
from skopt import gp_minimize, gbrt_minimize 
from skopt.plots import plot_convergence, plot_objective
from rl4greencrab import TwoActNormalized, twoActEnv

## Simulate agent actions in the environment

In [9]:
obs = "count-biomass-time"
config = {
    "w_mort_scale" : 600,
    "growth_k": 0.70,
    'random_start':True,
    'var_penalty_const': 0,
    'observation_type': obs,
    'control_randomness': True
}

In [10]:
dir_name = 'greencrab_two_act_env/count-biomass-time'

### Constant Action

In [23]:
evalEnv =  twoActEnv(config)
x = [1442.0569452909665, 366.1851707583983]
constant_agent = multiConstAction(env=evalEnv, action=np.array(x))
const_plot_agent = plot_agent(env_sim_df=None, 
                            agent_name=f'{dir_name}/const_agent_size', 
                            env=evalEnv, 
                            agent=constant_agent, 
                            save_dir='.')
df = const_plot_agent.gen_env_sim_df(rep=500, obs_names=['crabs'])
const_plot_agent.save_df(const_plot_agent.env_simulation_df, 'const_agent_sim_500')

/opt/conda/lib/python3.12/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [24]:
np.mean(df[df['t']==99]['rew'])

-6.5006536883449915

### RL Policy

In [21]:
evalEnv = TwoActNormalized(config)
agent_dir = "../models"
tqcAgent = TQC.load(f"{agent_dir}/TQC_noVar_2obs", device="cpu")
tqc_plot_agent = plot_agent(env_sim_df=None, 
                            agent_name=f'{dir_name}/tqc', 
                            env=evalEnv, 
                            agent=tqcAgent, 
                            save_dir='.')
df = tqc_plot_agent.gen_env_sim_df(rep=500, obs_names=['crabs'])
tqc_plot_agent.save_df(tqc_plot_agent.env_simulation_df, 'tqc_sim_500')

In [22]:
np.mean(df[df['t']==99]['rew'])

-4.252223012042923